# Mestrado em Computação Aplicada • ConvNeXt-Tiny (Versão Otimizada)
Classificação de imagens de grandes peixes migratórios — **mesmos splits da execução de 11/10/2026** (4.354 treino / 933 validação / 933 teste) para comparação justa no paper.

### Melhorias aplicadas em relação ao baseline (72%)
1. **Preservação de proporção (`resize_with_pad`)**: evita achatar o corpo e o rostro/nadadeiras de peixes alongados (ex.: *Tetrapturus pfluegeri*, marlins, atuns).
2. **Resolução maior (`320×320`, ajustável para `384×384` na A100)**: preserva detalhes taxonômicos finos (nadadeira dorsal, peitoral, rostro e linha lateral).
3. **Substituição da `Lambda` por `layers.Rescaling(255.0)`**: gera um `.keras` 100% compatível com `keras.models.load_model()` (pronto para colocar em `models/modelo.keras` no **FishID Studio / Cloud Run**).
4. **Fine-tuning com Warmup + CosineDecay + `AdamW` (`weight_decay=1e-2`, `clipnorm=1.0`)**: evita queda precoce do *learning rate* e carrega o melhor checkpoint da Fase A antes de iniciar a Fase B.
5. **Data Augmentation de campo + MixUp leve (`alpha=0.2`)**: maior robustez a iluminação solar, reflexo, ângulo e fundo de barco.
6. **Test-Time Augmentation (TTA)** na avaliação: reporta métricas **sem TTA** e **com TTA** para tabela comparativa do paper.


In [ ]:
# =============================================================================
# 1. Configurações principais do experimento otimizado
# =============================================================================
MODO = "treinar"  # "treinar", "analisar" ou "resultados"
MODELO = "convnext_tiny_otimizado"
SEED = 42

# Resolução e preservação morfológica
IMG_SIZE = (320, 320)          # Na A100 você pode testar até (384, 384)
PRESERVAR_PROPORCAO = True     # True usa resize_with_pad (crucial para peixes alongados)
BATCH_SIZE = 32                # Use 16 se subir para (384, 384) em GPU T4
USAR_MIXED_PRECISION = True    # Acelera bastante na A100/L4/T4

# Estratégia de treinamento em 2 fases
EPOCHS_HEAD = 12
EPOCHS_FINE = 35
LR_HEAD = 5e-4
LR_FINE = 5e-5                 # Pico após warmup (antes era 1e-5 fixo)
WEIGHT_DECAY = 1e-2            # Padrão do paper original do ConvNeXt
DROPOUT_RATE = 0.35
ULTIMAS_CAMADAS_FINE_TUNE = 95 # Libera estágios 2 e 3 (ou None para liberar tudo)

# Regularização e avaliação
USAR_MIXUP = True
MIXUP_ALPHA = 0.20
USAR_TTA = True                # Média entre foto original e espelhada no teste
GERAR_GRADCAM = True
N_GRADCAM = 6
ESPECIE_GRADCAM = None         # Ex.: "Tetrapturus pfluegeri"


## 2. Conectar ao Drive e carregar metadados da execução original (baseline)


In [ ]:
from pathlib import Path
from google.colab import drive
from IPython.display import Image, display
import json
import pandas as pd

if MODO not in {"resultados", "analisar", "treinar"}:
    raise ValueError('Escolha "resultados", "analisar" ou "treinar".')

drive.mount("/content/drive")
RESULTS_ROOT = Path("./data/experimentos")
origem = RESULTS_ROOT / "baseline_run"
resumo = json.loads((origem / "resumo_experimento.json").read_text(encoding="utf-8"))
resultados_salvos = pd.DataFrame(resumo["results"])
print("Baseline original (baseline_run):")
display(resultados_salvos.loc[resultados_salvos["model"].eq("convnext_tiny")].reset_index(drop=True))
cm_baseline = origem / "convnext_tiny" / "confusion_matrix.png"
if cm_baseline.exists():
    display(Image(filename=str(cm_baseline), width=720))


## 3. Preparar splits originais, diagnóstico de balanceamento e pipeline `tf.data`


In [ ]:
if MODO != "resultados":
    from datetime import datetime
    import math
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, top_k_accuracy_score

    keras.utils.set_random_seed(SEED)
    gpus = tf.config.list_physical_devices("GPU")
    if USAR_MIXED_PRECISION and gpus:
        keras.mixed_precision.set_global_policy("mixed_float16")
    else:
        keras.mixed_precision.set_global_policy("float32")

    USE_CLASS_WEIGHT = resumo.get("use_class_weight", True)
    CLASS_WEIGHT_POWER = resumo.get("class_weight_power", 0.5)
    CLASS_WEIGHT_MAX = resumo.get("class_weight_max", 3.0)
    LABEL_SMOOTHING = resumo.get("label_smoothing", 0.1)
    AUTOTUNE = tf.data.AUTOTUNE

    indices = json.loads((origem / "class_indices.json").read_text(encoding="utf-8"))
    classes = sorted(indices, key=indices.get)
    num_classes = len(classes)
    if classes != resumo["classes"] or [indices[c] for c in classes] != list(range(num_classes)):
        raise ValueError("A ordem de classes diverge da execução original.")

    ausentes_log = []
    def carregar_split(nome):
        tabela = pd.read_csv(origem / f"split_{nome}.csv")
        if not {"path", "species", "label"} <= set(tabela.columns):
            raise ValueError(f"Colunas do split {nome} incompletas.")
        if tabela.empty or tabela["path"].isna().any() or tabela["path"].duplicated().any():
            raise ValueError(f"O split {nome} contém caminhos inválidos, vazios ou repetidos.")
        if not tabela["label"].eq(tabela["species"].map(indices)).all():
            raise ValueError(f"Rótulos inconsistentes no split {nome}.")
        # Verifica se os arquivos existem no Drive (evita NotFoundError no meio da época)
        existe_mask = tabela["path"].astype(str).map(lambda p: Path(p).is_file())
        n_faltando = int((~existe_mask).sum())
        if n_faltando > 0:
            print(f"[AVISO] Split '{nome}': {n_faltando} arquivo(s) não encontrado(s) no Drive e removido(s) com segurança.")
            for p_miss, sp_miss in zip(tabela.loc[~existe_mask, "path"], tabela.loc[~existe_mask, "species"]):
                ausentes_log.append({"split": nome, "species": sp_miss, "path": p_miss})
            tabela = tabela.loc[existe_mask].reset_index(drop=True)
        if tabela.empty:
            raise FileNotFoundError(f"Nenhum arquivo do split '{nome}' foi encontrado no Drive. Verifique o caminho do dataset.")
        return tabela

    train_df = carregar_split("train")
    val_df = carregar_split("validation")
    test_df = carregar_split("test")

    paths = [set(tabela["path"]) for tabela in (train_df, val_df, test_df)]
    if sum(map(len, paths)) != len(set.union(*paths)):
        raise ValueError("Existe sobreposição de caminhos entre os splits.")

    print(f"Treino: {len(train_df)} | Validação: {len(val_df)} | Teste: {len(test_df)}")
    print("TensorFlow:", tf.__version__, "| GPUs:", gpus, "| Policy:", keras.mixed_precision.global_policy().name)
    print("\nDistribuição por espécie no treino:")
    display(train_df["species"].value_counts().rename("n_treino").to_frame())

    # Augmentation realista para fotos de campo/embarcação (entrada em [0, 1])
    data_augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.10, fill_mode="reflect"),
        layers.RandomTranslation(0.08, 0.08, fill_mode="reflect"),
        layers.RandomZoom((-0.15, 0.15), fill_mode="reflect"),
        layers.RandomContrast(0.18),
        layers.RandomBrightness(0.15, value_range=(0.0, 1.0)),
    ], name="field_augmentation")

    def ler_imagem(path):
        img = tf.image.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        img = tf.image.convert_image_dtype(img, tf.float32)  # [0, 1]
        if PRESERVAR_PROPORCAO:
            img = tf.image.resize_with_pad(img, IMG_SIZE[0], IMG_SIZE[1], antialias=True)
        else:
            img = tf.image.resize(img, IMG_SIZE, antialias=True)
        img.set_shape((*IMG_SIZE, 3))
        return img

    def aplicar_mixup(images, labels, alpha=MIXUP_ALPHA):
        images = tf.cast(images, tf.float32)
        labels = tf.cast(labels, tf.float32)
        batch_size = tf.shape(images)[0]
        g1 = tf.random.gamma([batch_size], alpha, dtype=tf.float32)
        g2 = tf.random.gamma([batch_size], alpha, dtype=tf.float32)
        lam = g1 / (g1 + g2 + 1e-7)
        lam = tf.maximum(lam, 1.0 - lam)
        idx = tf.random.shuffle(tf.range(batch_size))
        lam_img = tf.reshape(lam, [batch_size, 1, 1, 1])
        lam_lbl = tf.reshape(lam, [batch_size, 1])
        mixed_images = images * lam_img + tf.gather(images, idx) * (1.0 - lam_img)
        mixed_labels = labels * lam_lbl + tf.gather(labels, idx) * (1.0 - lam_lbl)
        return mixed_images, mixed_labels

    def make_dataset(split_df, training=False, use_mixup=False):
        file_paths = split_df["path"].astype(str).to_numpy()
        labels = split_df["label"].astype("int32").to_numpy()
        ds = tf.data.Dataset.from_tensor_slices((file_paths, labels))
        if training:
            ds = ds.shuffle(buffer_size=len(split_df), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(
            lambda p, y: (ler_imagem(p), tf.one_hot(y, num_classes)),
            num_parallel_calls=AUTOTUNE,
        )
        if training:
            # Proteção extra contra JPEGs corrompidos ou oscilações do Google Drive FUSE
            ds = ds.ignore_errors(log_warning=True)
        ds = ds.batch(BATCH_SIZE)
        if training:
            ds = ds.map(
                lambda x, y: (tf.cast(tf.clip_by_value(data_augmentation(x, training=True), 0.0, 1.0), tf.float32), tf.cast(y, tf.float32)),
                num_parallel_calls=AUTOTUNE,
            )
            if use_mixup:
                ds = ds.map(aplicar_mixup, num_parallel_calls=AUTOTUNE)
        return ds.prefetch(AUTOTUNE)

    if MODO == "treinar":
        train_ds = make_dataset(train_df, training=True, use_mixup=USAR_MIXUP)
        val_ds = make_dataset(val_df, training=False, use_mixup=False)
        RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        RUN_DIR = RESULTS_ROOT / f"run_{RUN_ID}_{MODELO}"
        RUN_DIR.mkdir(parents=True, exist_ok=False)
        for nome, split in (("train", train_df), ("validation", val_df), ("test", test_df)):
            split.to_csv(RUN_DIR / f"split_{nome}.csv", index=False)
        (RUN_DIR / "class_indices.json").write_text(json.dumps(indices, ensure_ascii=False, indent=2), encoding="utf-8")
        # Exporta a lista exata de classes para colar no app.py do FishID Studio
        (RUN_DIR / "class_names_fishid.json").write_text(json.dumps(classes, ensure_ascii=False, indent=2), encoding="utf-8")
        if ausentes_log:
            pd.DataFrame(ausentes_log).to_csv(RUN_DIR / "arquivos_ausentes.csv", index=False)

        counts = train_df["label"].value_counts().sort_index()
        weights = len(train_df) / (num_classes * counts)
        weights = weights.pow(CLASS_WEIGHT_POWER).clip(upper=CLASS_WEIGHT_MAX)
        # Nota: com MixUp ativo, rótulos viram soft-labels; desativamos class_weight se MixUp estiver ligado
        class_weight = {int(k): float(v) for k, v in weights.items()} if (USE_CLASS_WEIGHT and not USAR_MIXUP) else None
        saida = RUN_DIR / MODELO
    else:
        saida = origem / "analise_individual" / MODELO

    saida.mkdir(parents=True, exist_ok=True)


## 4. Arquitetura limpa (sem `Lambda`, pronta para FishID Studio) e Fine-Tuning com CosineDecay


In [ ]:
if MODO != "resultados":
    def construir_modelo(weights="imagenet"):
        inputs = keras.Input(shape=(*IMG_SIZE, 3), name="image")
        # Camada nativa Rescaling(255.0) substitui Lambda -> 100% compatível com keras.models.load_model()
        # No FishID Studio (app.py), mantenha NORMALIZE_DIV_255 = True para enviar [0, 1] aqui.
        x = layers.Rescaling(255.0, name="to_255_scale")(inputs)
        base = tf.keras.applications.ConvNeXtTiny(
            include_top=False,
            include_preprocessing=True,
            weights=weights,
            input_tensor=x,
        )
        base.trainable = False
        x = layers.GlobalAveragePooling2D(name="gap")(base.output)
        x = layers.LayerNormalization(epsilon=1e-6, name="head_ln")(x)
        x = layers.Dropout(DROPOUT_RATE, name="dropout")(x)
        # dtype='float32' na saída garante estabilidade numérica com mixed_float16
        outputs = layers.Dense(num_classes, activation="softmax", dtype="float32", name="predictions")(x)
        model = keras.Model(inputs, outputs, name="ConvNeXtTiny_FishID")
        return model, base

if MODO == "treinar":
    steps_per_epoch = math.ceil(len(train_df) / BATCH_SIZE)

    def build_cosine_schedule(peak_lr, epochs, warmup_epochs=2):
        total_steps = max(1, epochs * steps_per_epoch)
        warmup_steps = min(total_steps // 4, warmup_epochs * steps_per_epoch)
        return keras.optimizers.schedules.CosineDecay(
            initial_learning_rate=peak_lr * 0.1,
            decay_steps=max(1, total_steps - warmup_steps),
            warmup_target=peak_lr,
            warmup_steps=warmup_steps,
            alpha=0.02,
        )

    def compile_model(model, lr_schedule):
        model.compile(
            optimizer=keras.optimizers.AdamW(
                learning_rate=lr_schedule,
                weight_decay=WEIGHT_DECAY,
                clipnorm=1.0,
            ),
            loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
            metrics=[
                "accuracy",
                keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_accuracy"),
            ],
        )
        return model

    def make_callbacks(stage_dir, patience=10):
        stage_dir.mkdir(parents=True, exist_ok=True)
        return [
            keras.callbacks.ModelCheckpoint(
                filepath=str(stage_dir / "best.keras"),
                monitor="val_accuracy",
                mode="max",
                save_best_only=True,
                verbose=1,
            ),
            keras.callbacks.EarlyStopping(
                monitor="val_accuracy",
                mode="max",
                patience=patience,
                restore_best_weights=True,
                verbose=1,
            ),
            keras.callbacks.CSVLogger(str(stage_dir / "history.csv")),
        ]

    modelo, base_modelo = construir_modelo(weights="imagenet")

    # ---------------- FASE A: Treino da cabeça ----------------
    print("\n===== FASE A: Cabeça classificadora (backbone congelado) =====")
    head_dir = RUN_DIR / f"{MODELO}_head"
    compile_model(modelo, build_cosine_schedule(LR_HEAD, EPOCHS_HEAD, warmup_epochs=1))
    history_head = modelo.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_HEAD,
        callbacks=make_callbacks(head_dir, patience=5),
        class_weight=class_weight,
    )
    # Garante partir exatamente dos melhores pesos da Fase A
    best_head_path = head_dir / "best.keras"
    if best_head_path.exists():
        modelo.load_weights(str(best_head_path))

    # ---------------- FASE B: Fine-tuning real ----------------
    print("\n===== FASE B: Fine-tuning do ConvNeXt-Tiny =====")
    base_modelo.trainable = True
    if ULTIMAS_CAMADAS_FINE_TUNE is not None:
        for layer in base_modelo.layers[:-ULTIMAS_CAMADAS_FINE_TUNE]:
            layer.trainable = False
        print(f"Liberadas as últimas {ULTIMAS_CAMADAS_FINE_TUNE} de {len(base_modelo.layers)} camadas.")
    else:
        print(f"Backbone inteiro liberado ({len(base_modelo.layers)} camadas).")

    fine_dir = RUN_DIR / f"{MODELO}_finetune"
    compile_model(modelo, build_cosine_schedule(LR_FINE, EPOCHS_FINE, warmup_epochs=3))
    history_fine = modelo.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_FINE,
        callbacks=make_callbacks(fine_dir, patience=10),
        class_weight=class_weight,
    )

    best_fine_path = fine_dir / "best.keras"
    if best_fine_path.exists():
        modelo.load_weights(str(best_fine_path))

    # Salva modelo final pronto para copiar como models/modelo.keras no FishID Studio
    modelo_final_path = saida / "modelo.keras"
    modelo.save(modelo_final_path)
    print("\nModelo pronto para o FishID Studio salvo em:", modelo_final_path)


## 5. Avaliação no conjunto de teste (sem TTA vs. com TTA) + Matriz de Confusão


In [ ]:
if MODO != "resultados":
    if MODO == "analisar":
        # Ajuste o caminho se quiser reavaliar um run otimizado específico
        modelo = keras.models.load_model(str(saida / "modelo.keras"), compile=False)

    def inferir_dataset(model, flip_horizontal=False):
        ds = tf.data.Dataset.from_tensor_slices(test_df["path"].astype(str).to_numpy())
        ds = ds.map(ler_imagem, num_parallel_calls=AUTOTUNE)
        if flip_horizontal:
            ds = ds.map(tf.image.flip_left_right, num_parallel_calls=AUTOTUNE)
        ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
        return model.predict(ds, verbose=1)

    probs_base = inferir_dataset(modelo, flip_horizontal=False)
    if USAR_TTA:
        probs_flip = inferir_dataset(modelo, flip_horizontal=True)
        probabilidades = 0.5 * (probs_base + probs_flip)
    else:
        probabilidades = probs_base

    reais = test_df["label"].to_numpy(dtype=int)
    etiquetas = list(range(num_classes))

    def avaliar(probs, tag):
        preds = probs.argmax(axis=1)
        rep = classification_report(reais, preds, labels=etiquetas, target_names=classes, output_dict=True, zero_division=0)
        return {
            "variante": tag,
            "accuracy": accuracy_score(reais, preds),
            "top3_accuracy": top_k_accuracy_score(reais, probs, k=3, labels=etiquetas),
            "macro_f1": rep["macro avg"]["f1-score"],
            "weighted_f1": rep["weighted avg"]["f1-score"],
        }, rep

    linhas_metricas = []
    m_base, rep_base = avaliar(probs_base, f"{MODELO} (sem TTA)")
    linhas_metricas.append(m_base)
    if USAR_TTA:
        m_tta, rep_final = avaliar(probabilidades, f"{MODELO} (com TTA)")
        linhas_metricas.append(m_tta)
    else:
        rep_final = rep_base

    tabela_metricas = pd.DataFrame(linhas_metricas)
    tabela_metricas.to_csv(saida / "metricas_teste.csv", index=False)
    print("\n===== Comparativo de Métricas no Teste (933 imagens) =====")
    display(tabela_metricas)

    por_especie = pd.DataFrame(rep_final).T
    por_especie.to_csv(saida / "classification_report.csv")
    print("\nDesempenho por espécie (ordenado do menor para o maior recall):")
    display(por_especie.loc[classes].sort_values("recall"))

    previstos = probabilidades.argmax(axis=1)
    predicoes = test_df.copy()
    predicoes["predita"] = [classes[i] for i in previstos]
    predicoes["confianca"] = probabilidades.max(axis=1)
    predicoes["acertou"] = previstos == reais
    predicoes.to_csv(saida / "predicoes.csv", index=False)

    cm = confusion_matrix(reais, previstos, labels=etiquetas)
    cm_df = pd.DataFrame(cm, index=classes, columns=classes)
    cm_df.to_csv(saida / "confusion_matrix.csv")

    fig, ax = plt.subplots(figsize=(11, 8))
    sns.heatmap(cm_df, annot=True, fmt="d", cmap="crest", ax=ax)
    ax.set(title=f"Matriz de confusão — {MODELO}", xlabel="Predito", ylabel="Real")
    ax.tick_params(axis="x", labelrotation=55)
    fig.tight_layout()
    fig.savefig(saida / "confusion_matrix.png", dpi=150)
    plt.show()
    plt.close(fig)
    print("Resultados salvos em:", saida)
    print("\nLista CLASS_NAMES para o app.py do FishID Studio:")
    print(json.dumps(classes, ensure_ascii=False, indent=2))


## 6. Diagnóstico Didático com Grad-CAM: Onde o modelo erra nos peixes de bico?

### Como interpretar os 3 painéis (guia rápido de pesquisa)
Para cada erro de alta confiança (especialmente em **peixes de bico** — família *Istiophoridae*, como *Tetrapturus*, *Makaira*, *Kajikia* e *Istiophorus* — ou no par mais confundido da matriz), esta célula compara lado a lado:
1. **Foto original + Top-3 probabilidades:** mostra se a espécie correta ficou em 2º lugar por margem estreita (falta de detalhe fino) ou se o modelo errou com convicção.
2. **Grad-CAM da Classe Prevista (ERRO):** regiões quentes (vermelho/amarelo) indicam *por que* a rede escolheu a espécie errada. Se o calor estiver no **barco, mão do pescador, água ou convés**, há viés de fundo (*shortcut learning* — sinal forte para usar recorte/segmentação).
3. **Grad-CAM da Classe Real (CORRETA):** mostra onde a rede encontraria evidência para a espécie verdadeira. Compare com o painel 2: se os dois mapas olham para o **rostro (bico), nadadeira dorsal ou peitoral**, o erro é de taxonomia fina (justifica maior resolução `384×384` e `resize_with_pad`).


In [ ]:
if MODO != "resultados" and GERAR_GRADCAM:
    # -------------------------------------------------------------------------
    # 6.1 Ranking automático dos pares de espécies mais confundidos
    # -------------------------------------------------------------------------
    pares = []
    for i_real, esp_real in enumerate(classes):
        for i_pred, esp_pred in enumerate(classes):
            if i_real != i_pred and cm[i_real, i_pred] > 0:
                pares.append({
                    "especie_real": esp_real,
                    "predita_errada": esp_pred,
                    "erros": int(cm[i_real, i_pred]),
                    "pct_da_especie_real": round(100.0 * cm[i_real, i_pred] / max(1, int(cm[i_real].sum())), 1),
                })
    pares_df = pd.DataFrame(pares).sort_values("erros", ascending=False).reset_index(drop=True)
    pares_df.to_csv(saida / "pares_mais_confundidos.csv", index=False)
    print("===== Top 10 pares de espécies mais confundidos no teste =====")
    display(pares_df.head(10))

    # -------------------------------------------------------------------------
    # 6.2 Extrator Grad-CAM compatível com ConvNeXt-Tiny + mixed_float16
    # -------------------------------------------------------------------------
    # Seleciona a última camada espacial 4D do modelo (saída do stage 3 do ConvNeXt)
    camada_alvo = next(
        l for l in reversed(modelo.layers)
        if hasattr(l, "output") and len(l.output.shape) == 4
    )
    idx_alvo = modelo.layers.index(camada_alvo)
    extrator_feat = keras.Model(modelo.inputs, camada_alvo.output, name="gradcam_extractor")
    camadas_cabeca = modelo.layers[idx_alvo + 1:]
    print(f"Camada espacial usada no Grad-CAM: {camada_alvo.name} | shape={camada_alvo.output.shape}")

    def calcular_gradcam(img_tensor, classe_idx):
        x_batch = tf.expand_dims(img_tensor, axis=0)
        feat = tf.cast(extrator_feat(x_batch, training=False), tf.float32)
        with tf.GradientTape() as tape:
            tape.watch(feat)
            z = feat
            for l in camadas_cabeca[:-1]:
                z = l(z, training=False)
            dense = camadas_cabeca[-1]
            # Usa o logit pré-softmax para evitar saturação do gradiente
            logits = tf.linalg.matmul(tf.cast(z, tf.float32), tf.cast(dense.kernel, tf.float32))
            if dense.use_bias:
                logits = tf.nn.bias_add(logits, tf.cast(dense.bias, tf.float32))
            score = logits[:, int(classe_idx)]
        grads = tape.gradient(score, feat)
        if grads is None:
            return np.zeros(feat.shape[1:3], dtype=np.float32), False
        pesos = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
        mapa = tf.nn.relu(tf.reduce_sum(feat * pesos, axis=-1))[0]
        pico = float(tf.reduce_max(mapa))
        if pico <= 1e-8:
            return np.zeros(mapa.shape, dtype=np.float32), False
        return (mapa / pico).numpy(), True

    def descrever_foco(mapa):
        h_m, w_m = mapa.shape
        yy, xx = np.unravel_index(int(np.argmax(mapa)), mapa.shape)
        vert = ["superior", "central", "inferior"][min(2, int(3 * yy / max(h_m, 1)))]
        horiz = ["esquerda", "centro", "direita"][min(2, int(3 * xx / max(w_m, 1)))]
        # Verifica se a energia está muito concentrada nas bordas (forte indício de viés de fundo)
        borda = np.ones_like(mapa, dtype=bool)
        borda[1:-1, 1:-1] = False
        pct_borda = float(mapa[borda].sum() / (mapa.sum() + 1e-8))
        alerta = " (atenção nas bordas/fundo!)" if pct_borda > 0.55 else ""
        return f"região {vert}-{horiz}{alerta}", round(pct_borda * 100.0, 1)

    # -------------------------------------------------------------------------
    # 6.3 Seleção inteligente de erros (prioriza peixes de bico se existirem)
    # -------------------------------------------------------------------------
    erros_df = predicoes[~predicoes["acertou"]].copy()
    if ESPECIE_GRADCAM is not None:
        erros_filtrados = erros_df[erros_df["species"] == ESPECIE_GRADCAM]
    else:
        # Detecta automaticamente peixes de bico (Istiophoridae / Xiphias) nos nomes científicos
        termos_bico = ("tetrapturus", "makaira", "kajikia", "istiophorus", "xiphias", "marlin", "agulh", "espad")
        mask_bico = erros_df["species"].str.lower().apply(lambda s: any(t in s for t in termos_bico))
        erros_filtrados = erros_df[mask_bico] if mask_bico.any() else erros_df

    # Prioriza os erros em que o modelo errou com MAIOR confiança (mais informativos!)
    erros_filtrados = erros_filtrados.sort_values("confianca", ascending=False)
    selecionados = list(erros_filtrados.index[:N_GRADCAM])

    if not selecionados:
        print("Nenhum erro encontrado para o filtro selecionado!")
    else:
        print(f"\nGerando comparativo Grad-CAM para os {len(selecionados)} erros mais confiantes...")
        fig, axs = plt.subplots(len(selecionados), 3, figsize=(15, 4.4 * len(selecionados)), squeeze=False)
        diagnosticos = []

        for row_idx, sample_idx in enumerate(selecionados):
            row = predicoes.loc[sample_idx]
            img_t = ler_imagem(row["path"])
            img_np = np.clip(img_t.numpy(), 0.0, 1.0)
            probs_i = probabilidades[sample_idx]
            top3_idx = np.argsort(probs_i)[::-1][:3]
            top3_txt = "\n".join([f"{k+1}º {classes[c]} ({probs_i[c]:.1%})" for k, c in enumerate(top3_idx)])

            c_pred = int(previstos[sample_idx])
            c_real = int(reais[sample_idx])
            mapa_pred, ok_pred = calcular_gradcam(img_t, c_pred)
            mapa_real, ok_real = calcular_gradcam(img_t, c_real)

            # Painel 1: Foto original + Top-3
            axs[row_idx, 0].imshow(img_np)
            axs[row_idx, 0].set_title(
                f"REAL: {classes[c_real]}\nERROU PARA: {classes[c_pred]} ({probs_i[c_pred]:.1%})\n{top3_txt}",
                fontsize=9.5,
                loc="left",
            )

            # Painel 2: Grad-CAM da classe ERRADA (prevista)
            foco_pred, borda_pred = descrever_foco(mapa_pred) if ok_pred else ("sem ativação", 0.0)
            axs[row_idx, 1].imshow(img_np)
            if ok_pred:
                up_pred = tf.image.resize(mapa_pred[..., None], IMG_SIZE).numpy()[..., 0]
                axs[row_idx, 1].imshow(up_pred, cmap="jet", alpha=0.42, vmin=0, vmax=1)
            axs[row_idx, 1].set_title(
                f"ONDE OLHOU PARA ERRAR (Predita)\n{classes[c_pred]} · {foco_pred}",
                fontsize=9.5,
                color="#B91C1C",
            )

            # Painel 3: Grad-CAM da classe CERTA (real)
            foco_real, borda_real = descrever_foco(mapa_real) if ok_real else ("sem ativação", 0.0)
            axs[row_idx, 2].imshow(img_np)
            if ok_real:
                up_real = tf.image.resize(mapa_real[..., None], IMG_SIZE).numpy()[..., 0]
                axs[row_idx, 2].imshow(up_real, cmap="jet", alpha=0.42, vmin=0, vmax=1)
            axs[row_idx, 2].set_title(
                f"ONDE DEVERIA OLHAR (Real: {probs_i[c_real]:.1%})\n{classes[c_real]} · {foco_real}",
                fontsize=9.5,
                color="#0F766E",
            )

            for ax in axs[row_idx]:
                ax.axis("off")

            diagnosticos.append({
                "arquivo": Path(row["path"]).name,
                "real": classes[c_real],
                "predita_errada": classes[c_pred],
                "prob_predita": round(float(probs_i[c_pred]), 4),
                "prob_real": round(float(probs_i[c_real]), 4),
                "real_no_top3": bool(c_real in top3_idx),
                "foco_erro": foco_pred,
                "energia_borda_erro_pct": borda_pred,
                "foco_real": foco_real,
            })

        fig.suptitle(
            "Diagnóstico Grad-CAM de Erros — Predita Errada (vermelho) vs. Classe Real (verde-petróleo)",
            fontsize=13,
            fontweight="bold",
        )
        fig.tight_layout(rect=[0, 0, 1, 0.98])
        fig.savefig(saida / "gradcam_erros_comparativo.png", dpi=160, bbox_inches="tight")
        plt.show()
        plt.close(fig)

        diag_df = pd.DataFrame(diagnosticos)
        diag_df.to_csv(saida / "gradcam_diagnostico_erros.csv", index=False)
        print("\nResumo tabular dos erros analisados (salvo em gradcam_diagnostico_erros.csv):")
        display(diag_df)
